In [1]:
import json
from tqdm import tqdm
import logging

Настройка логгера, для вывода подробной информации работа скрипта в файл `log.log`

При этом основной вывод прогрессбара от `tqdm` будет осуществляться в `Out` ячейку

In [2]:
# Настройка логгера для записи в файл
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

# Создание обработчика для записи в файл
file_handler = logging.FileHandler('log.log')
file_handler.setLevel(logging.DEBUG)

# Создание форматтера для сообщений
FORMAT = "%(asctime)s %(levelname)s [%(process)s]: %(name)s: %(message)s"
formatter = logging.Formatter(FORMAT)
file_handler.setFormatter(formatter)

# Добавление обработчика к логгеру
logger.addHandler(file_handler)

Предполагается, что оригинальный датасет лежит в директории `../hf_deft_ru/`

In [3]:
# orig files
dev_file = "../hf_deft_ru/dev.json"
train_file = "../hf_deft_ru/train.json"

А преобразованный датасет сохраняем в директорию `../ls_deft_ru/`

In [4]:
# for label studio files
dev_file_ls = "../ls_deft_ru/dev.json"
train_file_ls = "../ls_deft_ru/train.json"

### Конвертер формата HF в LS (label studio)


**`ORIGINAL_DEFT`** - Имя "аннотатора", то есть которое будет отображаться как оригинальная разметка от DEFT.

In [14]:
def make_label_studio_obj(data: dict):
    """ Функция обработки одного элемента датасета.

    Создается элемент для добавления в Label Studio. 
    Преобразовываются к нужному виду все поля:
    - предложения на русском и на английском языках - к формату `data`,
    - спаны и связи - к формату `predictions`,
    - внутренний id датасета - к формату `meta`,
    - опционально: информация про связи, которые были убраны, т.к. связывали два спана из разных предложений - записаны в `meta`.
    
    :param data: загруженный словарь данных об элементе датасета.
    :return label_studio_obj: преобразованный к формату Label Studio объект данных.
    
    """
    # инициализируем множество для спанов
    span_ids = set()
    
    # Создаем объект для Label Studio
    label_studio_obj = {
        "meta": {
            "id": data["id"]
        },
        "data": {
            "text": data["text"],
            "text_rus": data.get("text_rus", "")
        },
        "predictions": [],
    }
    
    # Добавляем аннотации, если они есть
    if data.get("spans") or data.get("relations"):
        prediction = {
            "model_version": "ORIGINAL_DEFT",       # имя "аннотатора", то есть которое будет отображаться как уже размеченное
            "result": []
        }

        # Добавляем span-ы 
# "spans":[["0","39","Definition","T123"],["48","64","Term","T122"],["67","77","Definition-frag","T123-frag"]]
        for start, end, label, span_id in data.get("spans", []):
            start = int(start)
            end = int(end)
            span_obj = {
                "id": span_id,
                "from_name": "label",
                "to_name": "text",
                "type": "labels",
                "value": {
                    "start": start,
                    "end": end,
                    "text": data["text"][start:end],
                    "labels": [label]
                }
            }
            span_ids.add(span_id)
            prediction["result"].append(span_obj)

        # Добавляем отношения 
# "relations":[["T123-frag","T123","fragment"],["T123","T122","Direct-Defines"]]           
        for from_id, to_id, relation_type in data.get("relations", []):

            # Добавляем только в случае, если from_id и to_id принадлежат спанам из этого предложения
            # Иначе записываем информацию в meta и выводим в логах
            if from_id in span_ids and to_id in span_ids:
                relation_obj = {
                    "from_id": from_id,
                    "to_id": to_id,
                    "type": "relation",
                    "direction": "right",
                    "labels": [relation_type]
                }
                prediction["result"].append(relation_obj)
            else:
                relation_broken = {
                    "from_id": from_id,
                    "to_id": to_id,
                    "type": relation_type
                }
                logging.info(f"ID: '{data.get('id', 'no_id')}' has relation that connects different sentences. Relation info: {relation_broken}.")
                if 'broken_relations' not in label_studio_obj["meta"]:
                    label_studio_obj["meta"]["broken_relations"] = []
                label_studio_obj["meta"]["broken_relations"].append(relation_broken)
        
        label_studio_obj["predictions"].append(prediction)
        
        return label_studio_obj

In [15]:
def convert_hf2ls(hf_file, ls_file):
    """ Функция конвертации датасета HF (Hugging Face) в LS (Label Studio) формат

    :param hf_file: путь к файлу датасета в HF формате
    :param ls_file: путь к файлу для сохранения преобразованного датасета в LS формате
    
    """
    cnt_processed = 0
    cnt_added_in_ls = 0
    cnt_skip_span_relation = 0
    cnt_skip_text_rus = 0
    cnt_skip_all = 0


    with open(hf_file, 'r') as f:
        lines = f.readlines()
    
    label_studio_data = []
    
    for line in tqdm(lines):
        data = json.loads(line)
        cnt_processed += 1
        # 1. Проверка на наличие русского языка
        if "text_rus" in data and data["text_rus"] != "":
            # 2. Проверка на наличие спанов или связей
            if ( "spans" in data and data["spans"] ) or ( "relations" in data and data["relations"] ):
                label_studio_obj = make_label_studio_obj(data)
                label_studio_data.append(label_studio_obj)
                logging.info(f"ID: '{data.get('id', 'no_id')}' successfully added!")
                cnt_added_in_ls += 1
            else:
                logging.info(f"ID: '{data.get('id', 'no_id')}' has not spans or relations! Skip!")
                cnt_skip_span_relation +=1
                cnt_skip_all += 1
        else:
            logging.info(f"ID: '{data.get('id', 'no_id')}' has not text_rus or text_rus is empty! Skip!")
            cnt_skip_text_rus += 1
            cnt_skip_all += 1

    logging.info(f"Processed: {cnt_processed}, \tAdded: {cnt_added_in_ls}, \tSkip_text_rus: {cnt_skip_text_rus}, \tSkip_span_rel: {cnt_skip_span_relation}, \tSkip_total: {cnt_skip_all}.")
    
    with open(ls_file, 'w') as f:
        json.dump(label_studio_data, f, indent=4)

In [8]:
# convert_hf2ls("./input.json", "./output.json")

In [9]:
convert_hf2ls(dev_file, dev_file_ls)

 51%|█████▏    | 605/1179 [00:00<00:00, 6045.41it/s]

100%|██████████| 1179/1179 [00:00<00:00, 6195.50it/s]


In [10]:
convert_hf2ls(train_file, train_file_ls)

  3%|▎         | 670/24184 [00:00<00:03, 6695.17it/s]

100%|██████████| 24184/24184 [00:03<00:00, 7299.46it/s]


In [11]:
!grep "Processed:" log.log 

2024-09-23 18:34:32,412 INFO [108417]: root: Processed: 1179, 	Added: 332, 	Skip_text_rus: 0, 	Skip_span_rel: 847, 	Skip_total: 847.
2024-09-23 18:34:36,133 INFO [108417]: root: Processed: 24184, 	Added: 723, 	Skip_text_rus: 21179, 	Skip_span_rel: 2282, 	Skip_total: 23461.


#### Сбор ID предложений, где не создавалась связь при генерации конфига для LS



In [16]:
def collect_sentence_ids_of_broken_relations(input_file_ls):
    with open(input_file_ls, 'r') as f:
        label_studio_data = json.load(f)

    sent_ids = []
    for elem in label_studio_data:
        meta = elem["meta"]
        if "broken_relations" in meta.keys() and meta["broken_relations"]:
            sent_id = meta["id"]
            sent_ids.append(sent_id)
    
    return sent_ids

In [18]:
dev_sent_id_of_broken_relations = "../ls_deft_ru/dev_sent_id_of_broken_relations.json"
train_sent_id_of_broken_relations = "../ls_deft_ru/train_sent_id_of_broken_relations.json"

In [21]:
dev_sent_ids = collect_sentence_ids_of_broken_relations(dev_file_ls)
print(f"Amount sentence with broken relations: {len(dev_sent_ids)}")

with open(dev_sent_id_of_broken_relations, 'w') as f:
    json.dump(dev_sent_ids, f, indent=4)

Amount sentence with broken relations: 35


In [22]:
train_sent_ids = collect_sentence_ids_of_broken_relations(train_file_ls)
print(f"Amount sentence with broken relations: {len(train_sent_ids)}")

with open(train_sent_id_of_broken_relations, 'w') as f:
    json.dump(train_sent_ids, f, indent=4)

Amount sentence with broken relations: 93


### Конвертер выгружаемого файла Label studio с разметкой от разметчиков к HF формату

- спаны и отношения надо добавить в соответствующие `spans_rus` и `relations_rus` в HF

**Алгоритм работы:**
1. идем по списку из словарей (предложений)
2. в поле `annotations` список аннотаций. Скорее всего берем первую, так как у нас аннотатор только один будет (???). (Если длина annotations > 1, то вывожу id и смотрю)
3. аннотация - словарь. берем поле `result` - это список из span-ов или relation-ов
- учесть `choices` - если есть, то значит пропускаем и записываем id в какой нибудь список (пока реализовано просто пропуск, если есть `choices`)
4. потом по очереди берем по одному элементу из `result` (допустим, `res`)
5. если `res["type"] == "labels" and res['from_name'] == "label_rus" and res['to_name'] == "text_rus" and res["origin"] == "manual"`, то значит это русская разметка спана (лейбла).
- берем поле 'value', оттуда индексы "start" и "end", из списка 'labels' берем первый элемент - это название терма
- берем 'id'
- формируем список [ start, end, label, id ]
6. если `res["type"] == "relation"`, то значит это разметка связи.
- но тут может быть и англ связь и русс связь, поэтому проверяем по id, которые включает в себя связь. Они должны быть среди id из п.5. 
- берем из списка 'labels' берем первый элемент - это название связи
- берем 'from_id', 'to_id'
- формируем список [ from_id, to_id, label_relation_name ]
7. Составить соответствие HF датасета и выкачанного из Label Studio
- состыковать по `["meta"]["id"]`. т.е. внутренний ID HF датасета, который передавали в поле `meta` в LS
- загрузить датасет в список (каждый элемент - одна строка датасета, т.е. словарь)
- создать индекс соответствия `id -> элемент в списке` (т.е. к номеру элемента в списке)
- искать id из LS данных в индексе
- и добавлять пункты 5 и 6 как `spans_rus` и `relations_rus` соответственно
8. Добавляем пустые `spans_rus` и `relations_rus` туда, где их нет после п.7.

In [54]:
def convert_ls2hf(ls_file, hf_file):
    """ Функция конвертации размеченных данных LS (Label Studio) в HF (Hugging Face)  формат

    Спаны и отношения, размеченные в LS, добавляются в соответствующие `span_rus` и `relation_rus` в HF.
    
    :param ls_file: путь к файлу выгрузки размеченных данных в LS формате
    :param hf_file: путь к файлу датасета в HF формате, к которому необходимо добавить русскую разметку.

    """
    cnt_processed = 0
    cnt_added_in_hf = 0
    cnt_empty_added_in_hf = 0
    cnt_skip = 0
    cnt_not_in_orig_hf_file = 0



    # считываем данные из hf датасета
    hf_data = []        # list of dicts (dict = one line from hf dataset)
    with open(hf_file, 'r') as f:
        lines = f.readlines()
        for line in lines:
            hf_data.append(json.loads(line))
    
    # создаем индекс для быстрого доступа к нужному элементу HF по ID
    id2hfindex = {}
    for index, elem in tqdm(enumerate(hf_data), total=len(hf_data), desc="Create index id2hfindex"):
        id2hfindex[elem["id"]] = index

    # считываем данные с аннотациями (разметкой)
    with open(ls_file, 'r') as f:
        ls_data = json.load(f)
        
    for ls_elem in tqdm(ls_data, desc="Processing LS annotations"):
        cnt_processed += 1

        annotations = ls_elem.get("annotations", [])
        if len(annotations) != 1:
            logging.warning(f"Len annotations != 1. Check ID: '{ls_elem.get('id', 'no_id')}'.")
        annotation = annotations[0]

        SKIP_ANN = False                    # флаг для пропуска аннотации
        for res in annotation.get("result", []):
            if res["type"] == "choices":
                SKIP_ANN = True             # флаг устаравливаем в True, если встречается хоть одна "галочка"
                break
        # TODO: обрабатывать галочки и добавлять данные

        if SKIP_ANN:                         # переходим к следующему элементу LS, если флаг True
            cnt_skip += 1
            logging.info(f"LS. ID: '{ls_elem['meta']['id']}' has choices! Skip!")
            continue
        
        spans_rus = []
        span_rus_ids = set()
        relations_rus = []

        for res in annotation.get("result", []):
            
            try:
                if  res["type"] == "labels" and \
                        res['from_name'] == "label_rus" and \
                        res['to_name'] == "text_rus" and \
                        res["origin"] == "manual":
                    # это русская разметка спана (лейбла)
                    
                    start = str(res["value"]["start"])
                    end = str(res["value"]["end"])
                    label = res["value"]["labels"][0]
                    span_id = res["id"]

                    span = [ start, end, label, span_id ]
                    spans_rus.append(span)
                    
                    span_rus_ids.add(span_id)               # записываем в множество, для вычисления нужных связей в дальнейшем

                elif res["type"] == "relation":
                    # это связь (русс. или англ.)

                    from_id = res["from_id"]
                    to_id = res["to_id"]
                    label_relation = res['labels'][0]

                    # т.к. может быть и англ. связь и русс. связь, поэтому проверяем по id, которые включает в себя связь.
                    # Они должны быть среди id из span_rus_ids
                    if from_id in span_rus_ids and to_id in span_rus_ids:
                        relation = [ from_id, to_id, label_relation ]
                        relations_rus.append(relation)
            except Exception as e:
                logging.error(f"LS. ID: '{ls_elem['meta']['id']}' processing and have error!")
                raise(e)
        
        
        # добавление данных в HF датасет
        hf_id = ls_elem["meta"]["id"]

        if hf_id in id2hfindex:
            num_in_index = id2hfindex[hf_id]
            elem = hf_data[num_in_index] 
            elem["spans_rus"] = spans_rus
            elem["relations_rus"] = relations_rus
            hf_data[num_in_index] = elem

            cnt_added_in_hf += 1
        else:
            # в текущем HF файле нет предложения из LS 
            # значит предложение из другого HF файла
            cnt_not_in_orig_hf_file += 1

    # Добавить в HF датасет пустые поля spans_rus и relations_rus, если они не были добавлены
    for index, hf_elem in tqdm(enumerate(hf_data), total=len(hf_data), desc="Adding empty fields spans_rus and relations_rus"):
        if hf_elem is None:
            print(f"Index of empty elem : '{index}'")
            continue

        added_empty = False
        if "spans_rus" not in hf_elem:
            hf_elem.update({"spans_rus": []})
            added_empty = True
        if "relations_rus" not in hf_elem:
            hf_elem.update({"relations_rus": []})
            added_empty = True
        if added_empty:
            cnt_empty_added_in_hf += 1

    logging.info(f"Processed: {cnt_processed}, \tAdded: {cnt_added_in_hf}, \tAdded_empty: {cnt_empty_added_in_hf}, \tSent_not_in_this_HF: {cnt_not_in_orig_hf_file}, \tSkip_total: {cnt_skip}, \tLen_hf_file: {len(hf_data)}.")
    
    # Записываем данные в HF датасет
            # базовый случай для всех данных
    new_dataset = []
    for hf_elem in tqdm(hf_data, total=len(hf_data), desc="Adding data to new HF dataset"):
        new_dataset.append(json.dumps(hf_elem))

    # если надо записать только те, где есть relations_rus или spans_rus - то раскоментировать
    # new_dataset = []
    # with open("../ls_deft_ru/train_sent_id_of_broken_relations.json", 'r') as f:
    #     train_id_broken_rels = json.load(f)
    # with open("../ls_deft_ru/dev_sent_id_of_broken_relations.json", 'r') as f:
    #     dev_id_broken_rels = json.load(f)    
    # for hf_elem in tqdm(hf_data, total=len(hf_data), desc="Adding data to new HF dataset, only where have relations_rus or spans_rus"):
    #     if "spans_rus" in hf_elem and hf_elem["spans_rus"] or "relations_rus" in hf_elem and hf_elem["relations_rus"]:
    #         sent_id = hf_elem["id"]
    #         if sent_id not in train_id_broken_rels and sent_id not in dev_id_broken_rels:
    #             new_dataset.append(json.dumps(hf_elem))
            # new_dataset.append(json.dumps(hf_elem))



    with open(hf_file, 'w', encoding='utf-8') as f:
        f.write('\n'.join(new_dataset))

In [55]:
# ls_annotations = "../ls_annotations/pretty.json"
ls_annotations = "../ls_annotations/project-2-at-2024-09-30-09-44-9175de58.json"

In [56]:
# dev_file_hf = "../hf_deft_ru_with_970_sent_minus_choices/dev.json"
# train_file_hf = "../hf_deft_ru_with_970_sent_minus_choices/train.json"

dev_file_hf = "../hf_deft_ru/dev_with_rus.json"
train_file_hf = "../hf_deft_ru/train_with_rus.json"

In [57]:
# dev_file_ls_new = "../ls_deft_ru/dev_copy_new.json"
# train_file_ls_new = "../ls_deft_ru/train_copy_new.json"

In [58]:
convert_ls2hf(ls_file=ls_annotations, hf_file=dev_file_hf)

Create index id2hfindex: 100%|██████████| 1179/1179 [00:00<00:00, 303491.13it/s]


Adding empty fields spans_rus and relations_rus: 100%|██████████| 1179/1179 [00:00<00:00, 629145.60it/s]
Adding data to new HF dataset, only where have relations_rus or spans_rus: 100%|██████████| 1179/1179 [00:00<00:00, 83569.95it/s]


In [59]:
convert_ls2hf(ls_file=ls_annotations, hf_file=train_file_hf)

Adding empty fields spans_rus and relations_rus: 100%|██████████| 24184/24184 [00:00<00:00, 376073.97it/s]
Adding data to new HF dataset, only where have relations_rus or spans_rus: 100%|██████████| 24184/24184 [00:00<00:00, 364076.71it/s]


In [61]:
!grep "Processed:" log.log 

2024-09-23 18:34:32,412 INFO [108417]: root: Processed: 1179, 	Added: 332, 	Skip_text_rus: 0, 	Skip_span_rel: 847, 	Skip_total: 847.
2024-09-23 18:34:36,133 INFO [108417]: root: Processed: 24184, 	Added: 723, 	Skip_text_rus: 21179, 	Skip_span_rel: 2282, 	Skip_total: 23461.
2024-09-24 18:04:49,362 INFO [140077]: root: Processed: 8, 	Added: 8, 	Added_empty: 1171, 	Skip_total: 0.
2024-09-24 18:11:45,130 INFO [140077]: root: Processed: 8, 	Added: 8, 	Added_empty: 1171, 	Skip_total: 0, 	Len_hf_file: 1179.
2024-09-24 18:14:06,127 INFO [140077]: root: Processed: 8, 	Added: 8, 	Added_empty: 1171, 	Skip_total: 0, 	Len_hf_file: 1179.
2024-09-24 18:16:43,637 INFO [140077]: root: Processed: 8, 	Added: 8, 	Added_empty: 1171, 	Skip_total: 0, 	Len_hf_file: 1179.
2024-09-24 18:18:23,136 INFO [140077]: root: Processed: 8, 	Added: 8, 	Added_empty: 1171, 	Skip_total: 0, 	Len_hf_file: 1179.
2024-09-24 18:20:56,135 INFO [140077]: root: Processed: 8, 	Added: 8, 	Added_empty: 1171, 	Skip_total: 0, 	Len_hf_fi

In [53]:
# !cp ../hf_deft_ru/dev.json ../hf_deft_ru/dev_with_rus.json
# !cp ../hf_deft_ru/train.json ../hf_deft_ru/train_with_rus.json